# PHASE 4: Modeling (Regression & Clustering)
This notebook solves the core business problems using R.

In [ ]:
# install.packages(c("caret", "randomForest", "dplyr"))
library(dplyr)
library(readr)
library(ggplot2)

### 1. Bài toán Định vị (Clustering - K-Means)

In [ ]:
# Tải dữ liệu model
model_data <- read_csv('../../data/airbnb_model_ready.csv', show_col_types = FALSE)

# Chọn các đặc trưng: giá, vĩ độ, kinh độ, số lượng review
features <- model_data %>% select(price, latitude, longitude, number_of_reviews) %>% na.omit()

# Chuẩn hóa
scaled_features <- scale(features)

# Chạy K-Means với 4 cụm
set.seed(42)
kmeans_result <- kmeans(scaled_features, centers = 4, nstart = 10)

features$cluster <- as.factor(kmeans_result$cluster)

# Vẽ biểu đồ
ggplot(features, aes(x = longitude, y = latitude, color = cluster)) +
  geom_point(alpha = 0.5, size = 0.5) +
  theme_minimal() +
  labs(title = "Phân khúc phòng Airbnb (K-Means Clustering)")

### 2. Bài toán Định giá (Regression - Random Forest)

In [ ]:
library(randomForest)

# Rút gọn dữ liệu để train nhanh hơn cho mẫu
set.seed(42)
df_reg <- model_data %>% 
  select(price, room_type, neighbourhood_group, accommodates, minimum_nights, has_reviews) %>%
  mutate(across(c(room_type, neighbourhood_group), as.factor)) %>%
  sample_n(10000) # Lấy 10,000 dòng test thử

# Chia tập Train/Test
train_idx <- sample(1:nrow(df_reg), 0.8 * nrow(df_reg))
train_data <- df_reg[train_idx, ]
test_data <- df_reg[-train_idx, ]

# Khởi tạo mô hình Random Forest
rf_model <- randomForest(price ~ ., data = train_data, ntree = 50, importance = TRUE)

# Dự đoán
predictions <- predict(rf_model, test_data)

# Đánh giá (MAE)
mae <- mean(abs(test_data$price - predictions))
cat("Mean Absolute Error (MAE):", mae, "\n")

# Đánh giá R-squared
rss <- sum((predictions - test_data$price) ^ 2)
tss <- sum((test_data$price - mean(test_data$price)) ^ 2)
rsq <- 1 - rss/tss
cat("R-squared (R2):", rsq, "\n")